# Setup

In [10]:
!curl -L -o ~/Code/eec-289a/urbansound8k.zip https://www.kaggle.com/api/v1/datasets/download/chrisfilo/urbansound8k

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 5747M  100 5747M    0     0  15.5M      0  0:06:10  0:06:10 --:--:-- 18.5M


In [99]:
# Imports
import os
import random
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import librosa
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("Actual device name:", torch.cuda.get_device_name(0))

device: cuda
Actual device name: NVIDIA GeForce RTX 5060 Ti


In [100]:
def setup_audio_dataset(base_path: Path, zip_name: str):
    """Extracts and flattens audio files from a zip if the destination is empty."""
    audio_dir = base_path / "extracted_audio"
    audio_dir.mkdir(parents=True, exist_ok=True)
    
    files = list(audio_dir.rglob("*.[wmop][afg][vc3]*")) # Quick glob for wav, mp3, ogg, flac
    if not files:
        zip_path = base_path / zip_name
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, "r") as z:
            counts = {}
            for f in [x for x in z.infolist() if not x.is_dir() and x.filename.lower().endswith(('.wav', '.mp3', '.ogg', '.flac'))]:
                if "__MACOSX" in f.filename: continue
                name = Path(f.filename).name
                counts[name] = counts.get(name, 0) + 1
                out_name = name if counts[name] == 1 else f"{Path(name).stem}_{counts[name]-1}{Path(name).suffix}"
                with z.open(f) as src, open(audio_dir / out_name, "wb") as dst:
                    dst.write(src.read())
        files = list(audio_dir.rglob("*"))
    print(f"Found {len(files)} files. Preview: {[f.name for f in files[:5]]}")
    return sorted(files)

audio_files = setup_audio_dataset(Path("~/Code/eec-289a").expanduser(), "urbansound8k.zip")

Found 8732 files. Preview: ['171478-9-0-28.wav', '177537-7-0-2.wav', '13230-0-0-28.wav', '89099-4-0-1.wav', '128465-1-0-6.wav']


# Audio Utils + Patch Extraction

In [101]:

def load_audio_as_mfcc(path: Path, n_mfcc: int = 40, sr: int = 22050, duration: float = 4.0):
    """
    Loads audio, pads/crops to a fixed duration, and returns an MFCC tensor.
    Result shape: (1, n_mfcc, time_steps)
    """
    y, _ = librosa.load(path, sr=sr, duration=duration)
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc) # n_mfcc acts like the 'Height' of our image
    mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-6) #normalize
    x = torch.from_numpy(mfcc).float().unsqueeze(0) #(1, n_mfcc, T)
    return x

def extract_patches_from_mfcc(mfcc: torch.Tensor, window_width: int, max_patches: int | None = None, device: torch.device = "cpu"):
    """
    mfcc: (1, n_mfcc, T) -> patches: (N, 1, n_mfcc, window_width).
    Treats the MFCC as an image and slices it along the time axis.
    """
    assert mfcc.ndim == 3  # (C, H, W)
    C, n_mfcc, T = mfcc.shape
    kernel_size = (n_mfcc, window_width)
    stride = (1, window_width // 2) # 50% overlap
    unfold = torch.nn.Unfold(kernel_size=kernel_size, stride=stride)
    patches = unfold(mfcc.unsqueeze(0)) # (1, 1, n_mfcc, T)
    patches = patches.transpose(1, 2).reshape(-1, C, n_mfcc, window_width) # Reshape to (N, C, H, W) -> (N, 1, n_mfcc, window_width)
    if max_patches is not None and patches.shape[0] > max_patches:
        idx = torch.randperm(patches.shape[0])[:max_patches]
        patches = patches[idx]
    return patches.to(device)


# Sampling Logic

## Non-Parametric Sampling

In [114]:
def compute_audio_frontier(known: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    """
    known, valid: (H, W) bool where H=MFCC_bins, W=Time
    frontier: unknown & valid & adjacent-to-known (strictly in time)
    """
    assert known.shape == valid.shape
    H, W = known.shape
    known_f = known.float().view(1, 1, H, W)
    # look only at temporal neighbors as we always use whole frequency band
    k = torch.tensor([[[[1, 0, 1]]]], device=known.device).float() # (out_channels, in_channels, height, width)
    # Count known neighbors along the time axis
    # Padding=(0, 1) pads only the width (time) dimension
    nb = F.conv2d(known_f, k, padding=(0, 1)).view(H, W)
    # Frontier = unknown frames that are valid and have a known neighbor in time
    frontier = (~known) & valid & (nb > 0)
    return frontier
    
def choose_seed_patch_index(patches: torch.Tensor, seed_fg_min: int = 0) -> int:
    if seed_fg_min > 0:
        # Sum over all dims except the batch (N) -> C, F, T
        fg = patches.abs().sum(dim=[1, 2, 3])
        valid_indices = torch.nonzero(fg >= seed_fg_min).flatten()
        if valid_indices.numel() > 0:
            return valid_indices[random.randint(0, valid_indices.shape[0] - 1)].item()
    return random.randint(0, patches.shape[0] - 1)

def choose_frontier_frame(frontier: torch.Tensor, known: torch.Tensor, window_t: int) -> int:
    """
    Finds the time-index (x) on the frontier with the most known temporal neighbors. It there is a tie, pick randomly.
    frontier/known: (Time,) bool
    """
    if not frontier.any(): return None
    
    # 1. Prepare the 1D kernel (Size: Out_channels, In_channels, Width)
    # A window of 'ones' counts how many neighbors are 'True' (1.0)
    kernel = torch.ones((1, 1, window_t), device=known.device)
    known_f = known.float().view(1, 1, -1)

    pad = window_t // 2
    nb = F.conv1d(known_f, kernel, padding="same").view(-1) #look @ last dim (t)
    frontier_counts = torch.where(frontier, nb, -1.0) #(t), non-frontier = -1

    max_val = torch.max(frontier_counts)
    candidate_indices = torch.where(frontier_counts == max_val)[0]

    choice = torch.randint(0, len(candidate_indices), (1,)).item()
    return int(candidate_indices[choice])
    
def sample_from_candidates(
    cand_idx: torch.Tensor,   # (M,)
    dist: torch.Tensor,       # (N,)
    dmin: torch.Tensor,       # scalar
    weighted: bool = True,
    h_mult: float = 0.3,
) -> int:
    if weighted:
        cand_dists = dist[cand_idx]
        probs = torch.exp(-(cand_dists - dmin) / (h_mult * dmin + 1e-6))
        idx = torch.multinomial(probs, num_samples=1).item()
        chosen_idx = cand_idx[idx].item()
    else:
        # Uniform random selection
        random_pos = torch.randint(0, len(cand_idx), (1,)).item()
        chosen_idx = cand_idx[random_pos].item()
    return int(chosen_idx)

def masked_ssd(tgt_patch: torch.Tensor, patches_flat: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """
    tgt_patch: (1, F, T)
    patches_flat: (N, 1, F*T)
    mask: (F*T) bool
    """
    # Flatten target and compare only at masked (known) positions
    tgt_flat = tgt_patch.reshape(1, -1) # (1, F*T)
    # SSD over the known spectral-temporal bins
    diff = (patches_flat[:, mask] - tgt_flat[:, mask]) #sub target from all patches
    return diff.square().sum(dim=-1) #return squared diff

## Sparse Manifold Transform

In [115]:
import torch
import numpy as np

def compute_smt_embeddings(I: torch.Tensor, basis: torch.Tensor, lambd: float = 0.1, num_iter: int = 50, eta: float = None) -> torch.Tensor:
    """
    Computes Sparse Manifold Transform embeddings (sparse codes) using FISTA.
    
    Args:
        I: Input features, shape (feature_dim, batch_size)
        basis: SMT Dictionary / Basis, shape (feature_dim, M) where M is dict size.
        lambd: Sparsity penalty parameter.
        num_iter: Number of FISTA iterations.    
    Returns:
        ahat: The sparse embeddings, shape (M, batch_size)
    """
    N, batch_size = I.shape
    M = basis.shape[1]
    
    # Calculate step size (eta) if not provided based on the basis eigenvalues
    if eta is None:
        eigenvalues = torch.linalg.eigvalsh(torch.mm(basis.t(), basis))
        eta = 1.0 / eigenvalues.max().item()

    tk_n = 1.
    tk = 1.
    
    ahat = torch.zeros((M, batch_size), device=I.device)
    ahat_y = torch.zeros((M, batch_size), device=I.device)

    for t in range(num_iter):
        tk = tk_n
        tk_n = (1 + np.sqrt(1 + 4 * tk**2)) / 2
        ahat_pre = ahat
        
        # Res = I - D * ahat_y
        Res = I - torch.mm(basis, ahat_y)
        
        # ahat_y = ahat_y + eta * D^T * Res
        ahat_y = ahat_y.add(eta * torch.mm(basis.t(), Res))
        
        # Soft thresholding: ahat = max(ahat_y - eta * lambd, 0)
        ahat = ahat_y.sub(eta * lambd).clamp(min=0.)
        
        # Momentum update
        ahat_y = ahat.add(ahat.sub(ahat_pre).mul((tk - 1) / tk_n))
        
    return ahat

# Synthesis

In [116]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

def synthesize_audio_mfcc(
    patches: torch.Tensor,
    smt_basis: torch.Tensor,
    out_time_steps: int,
    R_SSL: float = 10.0,
    R_loc: float = 0.2,
    eps: float = 0.1,
    seed_fg_min: int = 0,
    weighted: bool = True,
    h_mult: float = 0.3
):
    # N: number of patches, C: channels, F_bins: freq, W_t: patch width
    N, C, F_bins, W_t = patches.shape
    
    # 1. Define Padding and Canvas
    pad_left = W_t // 2
    pad_right = W_t - pad_left - 1
    total_len = out_time_steps + pad_left + pad_right
    
    # Flatten patches for SSD: (N, C * F_bins * W_t)
    patches_flat = patches.reshape(N, -1)
    center_frames = patches[:, :, :, pad_left]

    # ---------------------------------------------------------
    # PRE-COMPUTE SOURCE EMBEDDINGS (SMT) - Pillar 3
    # ---------------------------------------------------------
    # Reshape for SMT: (Features, N)
    I_source = patches_flat.t() 
    # source_embeddings should result in (D, N) where D is basis dimension
    source_embeddings = compute_smt_embeddings(I_source, smt_basis, lambd=0.1, num_iter=50)
    
    # ---------------------------------------------------------
    # PRE-COMPUTE SOURCE TIME POSITIONS - Pillar 2
    # ---------------------------------------------------------
    source_time_positions = torch.linspace(0, 1, steps=N, device=patches.device)

    # Initialize Canvas
    out = torch.zeros((C, F_bins, total_len), device=patches.device)
    known = torch.zeros(total_len, dtype=torch.bool, device=patches.device)
    valid_mask = torch.zeros(total_len, dtype=torch.bool, device=patches.device)
    valid_mask[pad_left : pad_left + out_time_steps] = True

    chosen_patch_history = torch.full((total_len,), -1, dtype=torch.long, device=patches.device)

    # 2. Seed Placement
    seed_idx = choose_seed_patch_index(patches, seed_fg_min)
    cx = pad_left + out_time_steps // 2 # Start in the middle
    
    # Place initial seed patch
    out[:, :, cx - pad_left : cx + pad_right + 1] = patches[seed_idx]
    known[cx - pad_left : cx + pad_right + 1] = True
    chosen_patch_history[cx] = seed_idx

    pbar = tqdm(total=out_time_steps)
    pbar.update(known[valid_mask].sum().item())

    # 3. Synthesis Loop
    while known[valid_mask].sum() < out_time_steps:
        # Find frontier: neighbors of known frames that are within our valid output range
        kernel_adj = torch.tensor([1, 0, 1], dtype=torch.float32, device=patches.device).view(1, 1, 3)
        nb_adj = F.conv1d(known.float().view(1, 1, -1), kernel_adj, padding=1).view(-1)
        frontier = (~known) & valid_mask & (nb_adj > 0)

        if not frontier.any(): 
            break

        x = choose_frontier_frame(frontier, known, W_t)

        # ---------------------------------------------------------
        #LOW-LEVEL STATISTICS (SSD)
        # ---------------------------------------------------------
        tgt_patch = out[:, :, x - pad_left : x + pad_right + 1]
        mask_1d = known[x - pad_left : x + pad_right + 1]
        # Repeat mask to match flattened patch (C * F * W)
        mask_flat = mask_1d.repeat(C, F_bins, 1).reshape(-1)
        
        dist_ssd = masked_ssd(tgt_patch.reshape(-1), patches_flat, mask_flat)
        dmin = dist_ssd.min()
        mask_ssd = dist_ssd <= (1 + eps) * dmin
        
        # ---------------------------------------------------------
        # SPATIAL (TEMPORAL) NON-STATIONARITY 
        # ---------------------------------------------------------
        target_time_pos = (x - pad_left) / out_time_steps
        dist_loc = torch.abs(source_time_positions - target_time_pos)
        mask_loc = dist_loc <= R_loc
        
        # ---------------------------------------------------------
        # HIGH-LEVEL SEMANTICS (SMT)
        # ---------------------------------------------------------
        tgt_masked = tgt_patch.clone()
        tgt_masked[:, :, ~mask_1d] = 0.0
        I_tgt = tgt_masked.reshape(-1, 1)
        
        tgt_embedding = compute_smt_embeddings(I_tgt, smt_basis, lambd=0.1, num_iter=50)
        # Compute distance between (D, 1) and (D, N)
        dist_ssl = (source_embeddings - tgt_embedding).square().sum(dim=0)
        mask_ssl = dist_ssl <= R_SSL
        
        combined_mask = mask_ssd & mask_loc & mask_ssl
        cand_idx = torch.nonzero(combined_mask).flatten()
        
        if cand_idx.numel() == 0:
            # Fallback to SSD + Loc if SSL is too restrictive, then just SSD
            fallback = mask_ssd & mask_loc
            cand_idx = torch.nonzero(fallback).flatten()
            if cand_idx.numel() == 0:
                cand_idx = torch.nonzero(mask_ssd).flatten()

        chosen_idx = sample_from_candidates(cand_idx, dist_ssd, dmin, weighted, h_mult)
        
        # Update output
        out[:, :, x] = center_frames[chosen_idx]
        known[x] = True
        chosen_patch_history[x] = chosen_idx
        
        pbar.update(1)

    pbar.close()

    final_out = out[:, :, pad_left : pad_left + out_time_steps]
    final_history = chosen_patch_history[pad_left : pad_left + out_time_steps]

    return final_out, final_history

In [117]:
def reconstruct_audio(
    method: str,
    synth_mfcc: torch.Tensor = None,
    chosen_indices: torch.Tensor = None,
    source_audio: np.ndarray = None,
    hop_length: int = 512,
    window_t: int = None,
    sr: int = 22050,
    n_mels: int = 128
):
    """
    method: 'stitch' or 'griffin_lim'
    """
    if method == "stitch":
        if chosen_indices is None or source_audio is None or window_t is None:
            raise ValueError("'stitch' requires chosen_indices, source_audio, and window_t.")

        pad = window_t // 2
        stitched_audio = []

        for idx in chosen_indices:
            if idx == -1:
                # If a frame was left un-synthesized, insert silence
                stitched_audio.append(np.zeros(hop_length))
                continue

            # The patch index 'idx' corresponds to the start of the window.
            # The center frame of that patch is at 'idx + pad'.
            # We multiply by hop_length to get the raw audio sample indices.
            center_frame_idx = idx.item() + pad
            start_sample = center_frame_idx * hop_length
            end_sample = start_sample + hop_length

            # Extract the raw audio chunk
            audio_chunk = source_audio[start_sample:end_sample]

            # Handle edge cases if the chunk hits the end of the source audio
            if len(audio_chunk) < hop_length:
                audio_chunk = np.pad(audio_chunk, (0, hop_length - len(audio_chunk)))

            stitched_audio.append(audio_chunk)

        # Concatenate all chunks sequentially
        return np.concatenate(stitched_audio)

    elif method == "griffin_lim":
        if synth_mfcc is None:
            raise ValueError("'griffin_lim' requires synth_mfcc.")

        # Convert tensor back to numpy array: shape (n_mfcc, time_steps)
        mfcc_np = synth_mfcc.squeeze(0).cpu().numpy()

        # librosa's inverse feature requires knowing how many Mel bands were
        # originally used (default is often 128)
        print("Running Griffin-Lim reconstruction (this may take a moment)...")
        audio_recon = librosa.feature.inverse.mfcc_to_audio(
            mfcc_np,
            n_mels=n_mels,
            hop_length=hop_length,
            sr=sr
        )
        return audio_recon

    else:
        raise ValueError("Method must be 'stitch' or 'griffin_lim'")

In [119]:
import random
import librosa
import soundfile as sf
import torch
import numpy as np
from pathlib import Path
from IPython.display import display, Audio, HTML
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize

# --- Configuration ---
NUM_SAMPLES_TO_TEST = 2
N_MFCC = 40
OUT_TIME_STEPS = 150   # How many frames to synthesize
SMT_DICT_SIZE = 128    # Number of basis vectors in the dictionary (M)
R_SSL = 5.0            # Semantic distance threshold (Pillar 3)
R_LOC = 0.2            # NEW: Temporal distance threshold (Pillar 2)

# Ensure output directory exists for saved files
OUTPUT_DIR = Path("generated_audio")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Pick a few random files from your dataset
sample_files = random.sample(audio_files, min(NUM_SAMPLES_TO_TEST, len(audio_files)))
print(f"Running synthesis on {len(sample_files)} files...\n" + "="*40)

for i, filepath in enumerate(sample_files):
    print(f"\nProcessing File {i+1}: {filepath.name}")

    # ---------------------------------------------------------
    # STEP 2: Extract Features using SMT functions
    # ---------------------------------------------------------
    feature_dict = extract_mel_and_mfcc(
        audio_path=filepath,
        n_mels=N_MFCC, 
        n_mfcc=N_MFCC,
        window_ms=20.0,
        hop_ms=10.0
    )
    
    # Dynamically grab the native sample rate and hop length
    SR = feature_dict["sr"]
    HOP_LENGTH = feature_dict["hop_length"]

    # Display Original Audio
    raw_audio, _ = librosa.load(filepath, sr=SR)
    display(HTML(f"<b>Original Audio:</b> {filepath.name}"))
    display(Audio(raw_audio, rate=SR))

    # ---------------------------------------------------------
    # STEP 3: Extract Patches & Convert to PyTorch
    # ---------------------------------------------------------
    # Returns numpy array: [n_patches, n_features, patch_frames]
    patches_np = extract_300ms_patches_from_feature_dict(
        feature_dict=feature_dict,
        feature_key="mfcc",
        patch_hop_frames=1
    )
    
    # Convert to PyTorch Tensor and add Channel Dimension -> [N, 1, F_bins, W_t]
    patches = torch.tensor(patches_np, dtype=torch.float32, device=device).unsqueeze(1)
    
    # Dynamically set WINDOW_T based on the 300ms patch calculation
    WINDOW_T = patches.shape[3]
    print(f"Extracted {patches.shape[0]} patches (Window size: {WINDOW_T} frames). Learning SMT Dictionary...")
    
    # ---------------------------------------------------------
    # STEP 3.5: LEARN SMT BASIS USING K-MEANS
    # ---------------------------------------------------------
    N_patches, C, F_bins, W_t = patches.shape
    patches_flat_np = patches.reshape(N_patches, -1).cpu().numpy()
    
    kmeans = MiniBatchKMeans(n_clusters=min(SMT_DICT_SIZE, N_patches), batch_size=256, n_init='auto', random_state=42)
    kmeans.fit(patches_flat_np)
    
    basis_np = kmeans.cluster_centers_.T
    basis_np = normalize(basis_np, axis=0, norm='l2')
    smt_basis = torch.tensor(basis_np, dtype=torch.float32, device=device)

    print(f"Dictionary learned with shape {smt_basis.shape}. Synthesizing...")

    # ---------------------------------------------------------
    # STEP 4: Synthesize New MFCC
    # ---------------------------------------------------------
    out_mfcc, history_indices = synthesize_audio_mfcc(
        patches=patches,
        smt_basis=smt_basis,       
        out_time_steps=OUT_TIME_STEPS,
        R_SSL=R_SSL,
        R_loc=R_LOC,               # <-- NEW: Pass Pillar 2 threshold here
        eps=0.1,
        seed_fg_min=0,
        weighted=True
    )

    # ---------------------------------------------------------
    # STEP 5 & 6: Reconstruct Audio
    # ---------------------------------------------------------
    print("Reconstructing via Patch Stitching...")
    audio_stitched = reconstruct_audio(
        method="stitch",
        chosen_indices=history_indices,
        source_audio=raw_audio,
        hop_length=HOP_LENGTH,
        window_t=WINDOW_T
    )

    stitch_path = OUTPUT_DIR / f"stitched_{filepath.stem}.wav"
    sf.write(stitch_path, audio_stitched, SR)
    display(HTML(f"<b>Generated Audio (Stitched):</b> Saved to {stitch_path.name}"))
    display(Audio(audio_stitched, rate=SR))

    print("Reconstructing via Griffin-Lim...")
    audio_gl = reconstruct_audio(
        method="griffin_lim",
        synth_mfcc=out_mfcc,
        hop_length=HOP_LENGTH,
        sr=SR
    )

    gl_path = OUTPUT_DIR / f"griffin_lim_{filepath.stem}.wav"
    sf.write(gl_path, audio_gl, SR)
    display(HTML(f"<b>Generated Audio (Griffin-Lim):</b> Saved to {gl_path.name}"))
    display(Audio(audio_gl, rate=SR))

    print("="*40)

Running synthesis on 2 files...

Processing File 1: 84249-9-0-6.wav


Extracted 372 patches (Window size: 30 frames). Learning SMT Dictionary...
Dictionary learned with shape torch.Size([1200, 128]). Synthesizing...



100%|████████████████████████████████████████| 150/150 [00:00<00:00, 317.31it/s]

Reconstructing via Patch Stitching...


 20%|████████▍                                 | 30/150 [03:22<13:29,  6.74s/it]


...


Processing File 2: 155263-2-0-22.wav


Extracted 372 patches (Window size: 30 frames). Learning SMT Dictionary...
Dictionary learned with shape torch.Size([1200, 128]). Synthesizing...


100%|████████████████████████████████████████| 150/150 [00:00<00:00, 334.65it/s]

Reconstructing via Patch Stitching...


Reconstructing via Griffin-Lim...
Running Griffin-Lim reconstruction (this may take a moment)...
